# Feature Engineering in TuiML

This tutorial covers the feature engineering capabilities in TuiML, including:
- **Feature Selection** - Select the most relevant features
- **Feature Extraction** - Reduce dimensionality (PCA, Random Projection)
- **Feature Generation** - Create new features from existing ones

In [1]:
# Common imports
import numpy as np
from tuiml.datasets import load_iris, load_diabetes
from tuiml.evaluation import train_test_split, accuracy_score, r2_score
from tuiml.algorithms.trees import RandomForestClassifier

## 1. Feature Selection

Select the most relevant features to improve model performance and reduce overfitting.

### 1.1 Univariate Feature Selection (SelectKBestSelector)

Select the k highest-scoring features based on a scoring function.

In [2]:
from tuiml.features.selection import SelectKBestSelector, SelectPercentileSelector
from tuiml.evaluation import information_gain
from tuiml.evaluation.metrics import chi2

# Load data
X, y = load_iris()
print(f"Original shape: {X.shape}")

# Select top 2 features using information gain
selector = SelectKBestSelector(score_func=information_gain, k=2)
X_selected = selector.fit_transform(X, y)

print(f"Selected shape: {X_selected.shape}")
print(f"Feature scores: {selector.scores_}")
print(f"Selected indices: {selector.get_support(indices=True)}")

Original shape: (150, 4)
Selected shape: (150, 1)
Feature scores: 2.3769910703170005
Selected indices: [0]


In [3]:
# Select top 50% of features
selector_pct = SelectPercentileSelector(score_func=chi2, percentile=50)
X_pct = selector_pct.fit_transform(X, y)

print(f"\nSelectPercentileSelector (50%):")
print(f"  Selected shape: {X_pct.shape}")
print(f"  Selected indices: {selector_pct.get_support(indices=True)}")


SelectPercentileSelector (50%):
  Selected shape: (150, 2)
  Selected indices: [2 3]


### 1.2 Variance Threshold

Remove features with low variance (near-constant features).

In [4]:
from tuiml.features.selection import VarianceThresholdSelector

# Add a near-constant feature
X_with_const = np.column_stack([X, np.ones(len(X)) * 0.5 + np.random.random(len(X)) * 0.001])
print(f"Shape with constant: {X_with_const.shape}")

# Remove low-variance features
var_selector = VarianceThresholdSelector(threshold=0.01)
X_var = var_selector.fit_transform(X_with_const)

print(f"After variance threshold: {X_var.shape}")
print(f"Variances: {var_selector.variances_}")

Shape with constant: (150, 5)
After variance threshold: (150, 4)
Variances: [6.81122222e-01 1.86750667e-01 3.09242489e+00 5.78531556e-01
 8.98590863e-08]


### 1.3 Correlation-based Feature Selection (CFS)

WEKA's CFS algorithm selects features that are highly correlated with the target but uncorrelated with each other.

In [5]:
from tuiml.features.selection import CFSSelector

# CFS automatically determines the optimal subset
cfs = CFSSelector()
X_cfs = cfs.fit_transform(X, y)

print("CFS Feature Selection:")
print(f"  Original features: {X.shape[1]}")
print(f"  Selected features: {X_cfs.shape[1]}")
print(f"  Selected indices: {cfs.get_support(indices=True)}")

CFS Feature Selection:
  Original features: 4
  Selected features: 2
  Selected indices: [2 3]


### 1.4 Sequential Feature Selection

Greedy forward or backward feature selection using model performance.

In [6]:
from tuiml.features.selection import SequentialFeatureSelector

# Forward selection
sfs = SequentialFeatureSelector(
    estimator=RandomForestClassifier(n_estimators=10, random_state=42),
    n_features_to_select=2,
    direction='forward'
)
X_sfs = sfs.fit_transform(X, y)

print("Sequential Forward Selection:")
print(f"  Selected features: {X_sfs.shape[1]}")
print(f"  Selected indices: {sfs.get_support(indices=True)}")

Sequential Forward Selection:
  Selected features: 2
  Selected indices: [0 3]


### 1.5 Wrapper-based Selection

Uses a classifier's performance to evaluate feature subsets.

In [7]:
from tuiml.features.selection import WrapperSelector

# Wrapper selection with cross-validation
wrapper = WrapperSelector(
    estimator=RandomForestClassifier(n_estimators=10, random_state=42),
    cv=3
)
X_wrapper = wrapper.fit_transform(X, y)

print("Wrapper Feature Selection:")
print(f"  Selected features: {X_wrapper.shape[1]}")
print(f"  Selected indices: {wrapper.get_support(indices=True)}")

Wrapper Feature Selection:
  Selected features: 2
  Selected indices: [2 3]


## 2. Feature Extraction

Transform features into a lower-dimensional space while preserving information.

### 2.1 Principal Component Analysis (PCA)

In [8]:
from tuiml.features.extraction import PCAExtractor
from tuiml.preprocessing import StandardScaler

# Load and standardize data
X, y = load_iris()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA with fixed number of components
pca = PCAExtractor(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("PCA (2 components):")
print(f"  Original shape: {X_scaled.shape}")
print(f"  Transformed shape: {X_pca.shape}")
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"  Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

PCA (2 components):
  Original shape: (150, 4)
  Transformed shape: (150, 2)
  Explained variance ratio: [0.72770452 0.23030523]
  Total variance explained: 95.80%


In [9]:
# PCA with variance threshold (keep 95% variance)
pca_95 = PCAExtractor(n_components=0.95)
X_pca_95 = pca_95.fit_transform(X_scaled)

print("\nPCA (95% variance):")
print(f"  Components needed: {X_pca_95.shape[1]}")
print(f"  Explained variance ratio: {pca_95.explained_variance_ratio_}")


PCA (95% variance):
  Components needed: 2
  Explained variance ratio: [0.72770452 0.23030523]


### 2.2 Random Projection

Fast dimensionality reduction using random matrices.

In [10]:
from tuiml.features.extraction import RandomProjectionExtractor, SparseRandomProjectionExtractor

# Gaussian random projection
rp = RandomProjectionExtractor(n_components=2, distribution='gaussian', random_state=42)
X_rp = rp.fit_transform(X_scaled)

print("Random Projection (Gaussian):")
print(f"  Transformed shape: {X_rp.shape}")

# Sparse random projection (faster for high-dimensional data)
sparse_rp = SparseRandomProjectionExtractor(n_components=2, random_state=42)
X_sparse = sparse_rp.fit_transform(X_scaled)

print("\nSparse Random Projection:")
print(f"  Transformed shape: {X_sparse.shape}")

Random Projection (Gaussian):
  Transformed shape: (150, 2)

Sparse Random Projection:
  Transformed shape: (150, 2)


## 3. Feature Generation

Create new features from combinations or transformations of existing features.

### 3.1 Polynomial Features

In [11]:
from tuiml.features.generation import PolynomialFeaturesGenerator, InteractionFeaturesGenerator

# Simple example
X_simple = np.array([[1, 2], [3, 4], [5, 6]])

# Polynomial features (degree 2)
poly = PolynomialFeaturesGenerator(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_simple)

print("Polynomial Features (degree=2):")
print(f"  Original: {X_simple.shape} -> Transformed: {X_poly.shape}")
print(f"  Feature names: {poly.get_feature_names_out(['x1', 'x2'])}")
print(f"  First sample: {X_simple[0]} -> {X_poly[0]}")

Polynomial Features (degree=2):
  Original: (3, 2) -> Transformed: (3, 5)
  Feature names: ['x1' 'x2' 'x1^2' 'x1*x2' 'x2^2']
  First sample: [1 2] -> [1 2 1 2 4]


In [12]:
# Interaction features only (no powers)
interact = InteractionFeaturesGenerator()
X_interact = interact.fit_transform(X_simple)

print("\nInteraction Features:")
print(f"  Original: {X_simple.shape} -> Transformed: {X_interact.shape}")
print(f"  Includes: original + pairwise interactions")


Interaction Features:
  Original: (3, 2) -> Transformed: (3, 3)
  Includes: original + pairwise interactions


### 3.2 Mathematical Transformations

In [13]:
from tuiml.features.generation import MathematicalFeaturesGenerator

# Positive data for log/sqrt
X_pos = np.abs(X[:5]) + 1

# Apply mathematical transformations
math_features = MathematicalFeaturesGenerator(transformations=['log1p', 'sqrt', 'square'])
X_math = math_features.fit_transform(X_pos)

print("Mathematical Features:")
print(f"  Original: {X_pos.shape} -> Transformed: {X_math.shape}")
print(f"  Transformations: log1p, sqrt, square for each feature")

Mathematical Features:
  Original: (5, 4) -> Transformed: (5, 16)
  Transformations: log1p, sqrt, square for each feature


### 3.3 Binning Features

In [14]:
from tuiml.features.generation import BinningFeaturesGenerator

# Create binned features
binner = BinningFeaturesGenerator(n_bins=3, strategy='quantile', encode='onehot')
X_binned = binner.fit_transform(X[:, :2])  # Use first 2 features

print("Binning Features:")
print(f"  Original shape: {X[:, :2].shape}")
print(f"  Binned shape (one-hot): {X_binned.shape}")

Binning Features:
  Original shape: (150, 2)
  Binned shape (one-hot): (150, 6)


## 4. Comparing Feature Selection Methods

Evaluate different feature selection methods on model performance.

In [15]:
from tuiml.evaluation import StratifiedKFold

X, y = load_iris()

# Define feature selection methods
methods = [
    ('All Features', None),
    ('SelectKBestSelector (k=2)', SelectKBestSelector(score_func=information_gain, k=2)),
    ('CFS', CFSSelector()),
    ('PCA (2 comp)', PCAExtractor(n_components=2)),
]

print("Feature Selection Comparison (5-fold CV):")
print("=" * 50)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, selector in methods:
    scores = []
    for train_idx, test_idx in kfold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # Apply feature selection
        if selector is not None:
            X_train_sel = selector.fit_transform(X_train, y_train)
            X_test_sel = selector.transform(X_test)
        else:
            X_train_sel, X_test_sel = X_train, X_test
        
        # Train and evaluate
        model = RandomForestClassifier(n_estimators=10, random_state=42)
        model.fit(X_train_sel, y_train)
        scores.append(accuracy_score(y_test, model.predict(X_test_sel)))
    
    scores = np.array(scores)
    n_features = X_train_sel.shape[1] if selector else X.shape[1]
    print(f"{name:<28}: {scores.mean():.2%} (+/- {scores.std() * 2:.2%}) [{n_features} features]")

Feature Selection Comparison (5-fold CV):
All Features                : 94.00% (+/- 2.67%) [4 features]
SelectKBestSelector (k=2)   : 66.67% (+/- 13.33%) [1 features]
CFS                         : 96.67% (+/- 4.22%) [2 features]
PCA (2 comp)                : 92.00% (+/- 5.33%) [2 features]


## 5. Feature Engineering Pipeline

Combine multiple feature engineering steps in a complete workflow.

In [16]:
from tuiml.preprocessing import StandardScaler

# Load data
X, y = load_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Original features: {X_train.shape[1]}")

# Step 1: Generate polynomial features
poly = PolynomialFeaturesGenerator(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
print(f"After polynomial expansion: {X_train_poly.shape[1]}")

# Step 2: Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_poly)
X_test_scaled = scaler.transform(X_test_poly)

# Step 3: Remove low-variance features
var_sel = VarianceThresholdSelector(threshold=0.01)
X_train_var = var_sel.fit_transform(X_train_scaled)
X_test_var = var_sel.transform(X_test_scaled)
print(f"After variance threshold: {X_train_var.shape[1]}")

# Step 4: Select best features
selector = SelectKBestSelector(score_func=information_gain, k=10)
X_train_final = selector.fit_transform(X_train_var, y_train)
X_test_final = selector.transform(X_test_var)
print(f"After SelectKBestSelector: {X_train_final.shape[1]}")

# Train and evaluate
model = RandomForestClassifier(n_estimators=20, random_state=42)
model.fit(X_train_final, y_train)
y_pred = model.predict(X_test_final)

print(f"\nFinal Accuracy: {accuracy_score(y_test, y_pred):.2%}")

Original features: 4
After polynomial expansion: 14
After variance threshold: 14
After SelectKBestSelector: 1

Final Accuracy: 70.00%
